In [ ]:
import json
import gzip
import tarfile
import io
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd

from plot_utils import plot_figure4, fig_json_hook

In [ ]:
FIGURE_OUT_PATH = "../../../data/output/figures/assets/figure_4.png"

# The ExCALIBR + REVEL data behind panels a, b, c, e and f now come from a full
# out-of-bag export. Panel d (illustrative simulation) and the genome-wide
# aggregate REVEL thresholds in panel e are not part of that export, so those
# few values are still pulled from the legacy combined blob below.
OLD_DATA_PATH = "figure4_data.json.gz"
EXCALIBR_ARCHIVE_PATH = "../../../data/input/mave_calibration/excalibr/fig4_data_oob.tar.gz"
ZIP_ROOT = "fig4_data_oob"
DATASET_NAME = "MSH2_Jia_2021_clinvar_2018"


def _aread(tf, name):
    """Read one member's raw bytes out of the ExCALIBR tar.gz export."""
    return tf.extractfile(f"{ZIP_ROOT}/{name}").read()


def _load_scoreset(tf, score_csv_name, snv_csv_name=None):
    """Rebuild the minimal Scoreset-like interface plot_utils needs
    (.scores, .sample_assignments, .sample_counts, .snv_scores) from the
    ExCALIBR score/sample-assignment export."""
    df = pd.read_csv(io.BytesIO(_aread(tf, score_csv_name)), compression="gzip")
    n_samples = max(int(i) for row in df["sample_assignments"] for i in str(row).split(",")) + 1
    assignments = np.zeros((len(df), n_samples), dtype=bool)
    for row_idx, row in enumerate(df["sample_assignments"]):
        for i in str(row).split(","):
            assignments[row_idx, int(i)] = True

    snv_scores = None
    if snv_csv_name is not None:
        snv_scores = pd.read_csv(
            io.BytesIO(_aread(tf, snv_csv_name)), compression="gzip"
        )["score"].to_numpy()

    return SimpleNamespace(
        scores=df["score"].to_numpy(),
        sample_assignments=assignments,
        sample_counts=assignments.sum(axis=0),
        snv_scores=snv_scores,
    )


with tarfile.open(EXCALIBR_ARCHIVE_PATH, "r:gz") as tf:
    # Data for 4a-b (ExCALIBR MSH2)
    MSH2_scoreset_2018 = _load_scoreset(tf, "scoreset_2018.csv.gz")
    MSH2_scoreset = _load_scoreset(tf, "scoreset_2025.csv.gz", "scoreset_2025_snv.csv.gz")

    calibration_summary_raw = json.loads(
        _aread(tf, f"calibration/{DATASET_NAME}/{DATASET_NAME}_2c_avg_calibration.json")
    )
    excalibr_calibration_summary = {
        "prior": calibration_summary_raw["prior"],
        "point_ranges": calibration_summary_raw["point_ranges"],
    }
    excalibr_n_c = calibration_summary_raw["n_c"]
    MSH2_scoreset_flipped = bool(calibration_summary_raw["scoreset_flipped"])
    MSH2_n_samples = int((MSH2_scoreset.sample_counts > 0).sum())

    bootstrap_fits_raw = json.loads(gzip.decompress(_aread(tf, "bootstrap_fits.json.gz")))
    excalibr_fits = [entry["2c"] for entry in bootstrap_fits_raw[DATASET_NAME].values()]

    # score_range is the pipeline's own shared score-axis grid for this
    # dataset/component (stored alongside the calibration json, not derived
    # from the bootstrap fits) -- read it directly rather than approximating
    # it from the fits' xlims, which differs at the margins (a uniform
    # 0.00648-wide grid here vs. a reconstructed np.linspace) and shifts bin
    # edges for every binwidth-based histogram downstream.
    lr_values_raw = json.loads(
        gzip.decompress(_aread(tf, f"calibration/{DATASET_NAME}/{DATASET_NAME}_2c_avg_lr_values.json.gz"))
    )
    MSH2_score_range = np.asarray(lr_values_raw["score_range"])

    # Data for 4c (ExCALIBR confusion matrices) - panel_c.json already carries the
    # aggregated (summed-over-datasets) confusion matrices, so each list needs only
    # that one entry: plot_panel_c sums whatever is in the list.
    panel_c_raw = json.loads(_aread(tf, "panel_c.json"))
    excalibr_oob_evidence = [pd.DataFrame(panel_c_raw["danz_agg"])]
    auth_assignments = [pd.DataFrame(panel_c_raw["auth_agg"])]
    datasets_considered = None  # unused by plot_figure4/plot_panel_c
    vus_pct_danz = panel_c_raw.get("vus_pct_danz")
    vus_pct_auth = panel_c_raw.get("vus_pct_auth")

    # Data for 4e (REVEL gene-specific vs. genome-wide aggregate comparison)
    gene_4e = "MSH2"
    dist_4e = "REVEL"
    labdat_4e = pd.read_csv(
        io.BytesIO(_aread(tf, "revel/MSH2_REVEL_labeled.txt.gz")),
        sep="\t", header=None, compression="gzip",
    )
    revel_scores_raw = pd.read_csv(
        io.BytesIO(_aread(tf, "revel/MSH2_REVEL_scores.tsv.gz")),
        sep="\t", header=None, compression="gzip",
    )
    snvdf_4e = pd.DataFrame({
        "REVEL": revel_scores_raw[7],
        "merg_clinvar_sig": "allSNVs",
        "GeneSymbol": gene_4e,
    })
    gene_thresholds = pd.read_csv(
        io.BytesIO(_aread(tf, "revel/REVEL_gene_specific_calibration_thresholds_20260118.csv.gz")),
        index_col=0, compression="gzip",
    )
    gene_specific_thresholds_4e = gene_thresholds.loc[gene_4e].drop("calibration_model")

    # Data for 4f (REVEL gene-specific vs. genome-wide heatmap)
    dist_4f = "REVEL"
    finalout_4f = pd.read_csv(
        io.BytesIO(_aread(tf, "revel/REVEL_heatmap_data_pillar.csv.gz")),
        index_col=0, compression="gzip",
    )

# Values not present in the ExCALIBR export: panel d's illustrative simulation
# and panel e's genome-wide (gene-aggregate) REVEL thresholds. Pulled from the
# legacy blob without the Scoreset-reconstructing object_hook, since only a
# handful of small values are needed here.
with gzip.open(OLD_DATA_PATH, "rt", encoding="utf-8") as f:
    old_raw = json.load(f)

prior = old_raw["prior"]
Post_p = fig_json_hook(old_raw["Post_p"])
Post_b = fig_json_hook(old_raw["Post_b"])
p_data = fig_json_hook(old_raw["p_data"])
b_data = fig_json_hook(old_raw["b_data"])
genome_wide_thresholds_4e = fig_json_hook(old_raw["oldsorted_thresholds_4e"])

In [ ]:
fig = plot_figure4(
    # Data for 4a-b (ExCALIBR MSH2)
    MSH2_scoreset_2018, MSH2_scoreset, excalibr_calibration_summary, excalibr_fits, MSH2_score_range, excalibr_n_c, MSH2_n_samples, MSH2_scoreset_flipped,
    # Data for 4c (ExCALIBR confusion matrices)
    excalibr_oob_evidence, auth_assignments, datasets_considered,
    # Data for 4d (gene-specific predictor calibration overview)
    prior, Post_p, Post_b, p_data, b_data,
    # Data for 4e (REVEL gene-specific vs. genome-wide aggregate comparison)
    gene_4e, dist_4e, labdat_4e, snvdf_4e, gene_specific_thresholds_4e, genome_wide_thresholds_4e,
    # Data for 4f (REVEL gene-specific vs. genome-wide heatmap)
    dist_4f, finalout_4f,
    vus_pct_danz=vus_pct_danz, vus_pct_auth=vus_pct_auth,
)

Path(FIGURE_OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_OUT_PATH,dpi=300,bbox_inches='tight')